# 02 · NASA POWER climate summaries and facts for every POWER cell

For every land cell of the climate universe (amendment A2 in `PLAN.md`):
- **standardised long-term climate summaries** (7 values: sunlight, growing degree days, heating and cooling
  degree days, low-wind low-sun days, May–September root-zone wetness and temperature, 2001–2025 means), used by
  "places like this point";
- the **summary-statistics vector** (yearly seasonal ranks, heat-day counts, rain totals, growing degree days and
  long-term means) and the **climate facts**, used only by comparison arm (a), the CLIP-style search.

These are plain POWER numbers; no learned climate embeddings. Standardisation uses cells outside the held-out states.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.climate_facts import fact_table, long_term_vector, summary_features
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.power import add_derived, fetch_daily
from askmap.universe import Universe

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
OUT = ROOT / "climate"
OUT.mkdir(exist_ok=True)
P = cfg["power"]
YEARS = list(range(int(P["start"][:4]), int(P["end"][:4]) + 1))
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
print(f"{len(cells)} cells, {int(cells.heldout.sum())} in held-out states")

849 cells, 124 in held-out states


In [2]:
lt = long_term_vector(uni, CAP, YEARS)
np.save(OUT / "long_term_z.npy", lt)
summary = summary_features(uni, CAP, YEARS)
np.save(OUT / "enc_summary.npy", summary)
fact_ids, table = fact_table(uni, CAP, YEARS)
np.savez_compressed(OUT / "facts.npz", ids=np.array(fact_ids), table=table)
fam = pd.Series([f.split("|")[0] for f in fact_ids])
info = {
    "cells": len(cells),
    "long_term": list(lt.shape),
    "summary": list(summary.shape),
    "facts": len(fact_ids),
    "minutes": round((time.time() - T_START) / 60, 1),
    "MB": {p.name: round(p.stat().st_size / 1e6, 2) for p in OUT.iterdir()},
}
(RES / "02_climate_summaries.json").write_text(json.dumps(info, indent=1))
print(info)
pd.DataFrame({"family": fam, "share_of_cells_true": table.mean(axis=0)}).groupby("family").describe().round(3)

{'cells': 849, 'long_term': [849, 7], 'summary': [849, 907], 'facts': 4006, 'minutes': 0.6, 'MB': {'facts.npz': 0.35, 'long_term_z.npy': 0.02, 'enc_summary.npy': 3.08}}


share_of_cells_true                                                 
                     count   mean    std    min    25%    50%    75%    max
family                                                                     
gdd                  125.0  0.370  0.303  0.006  0.121  0.220  0.691  0.941
heat                 900.0  0.537  0.305  0.000  0.297  0.580  0.800  0.993
hist                2700.0  0.160  0.161  0.000  0.035  0.117  0.237  0.955
long                  28.0  0.424  0.084  0.286  0.339  0.419  0.494  0.561
rain                 250.0  0.381  0.318  0.000  0.038  0.392  0.658  0.963
sun                    3.0  0.236  0.142  0.102  0.161  0.219  0.302  0.385

In [3]:
names = list(CAP["climate_long"]["variables"])
pd.DataFrame(lt, columns=names).assign(state=cells.state.values).groupby("state").mean().round(2)

,ALLSKY_SFC_SW_DWN,GDD10,HDD18,CDD18,LOW_WIND_SUN,GWETROOT,T2M
state,,,,,,,
,-0.90,-0.49,0.51,-0.27,-0.51,0.04,-0.70
AL,0.79,1.46,-1.55,1.63,-1.49,0.71,1.50
AR,0.26,0.79,-0.78,0.63,-0.55,1.29,0.76
CO,1.85,-0.07,-0.18,0.03,1.48,-1.04,0.10
DE,-0.42,0.43,-0.55,0.16,-0.93,1.07,0.35
FL,1.43,2.07,-2.05,2.61,-1.35,-0.28,1.91
GA,0.91,1.51,-1.58,1.69,-1.17,0.18,1.48
IA,-0.64,-0.44,0.53,-0.56,-0.55,0.30,-0.44
IL,-0.51,0.18,-0.02,0.03,-0.97,0.92,0.18
